In [1]:
from rich.traceback import install
from tqdm import tqdm
from omegaconf import OmegaConf
import wandb
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageShow 

In [2]:
# Import autoencoder utility tools
from mnist_autoencoders.data.utils import train_loader, test_loader, output_to_image
from mnist_autoencoders.models.VAE import VAE

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
install(show_locals=True)
cfg = OmegaConf.load('../configs/vae.yaml')
wandb.init(project="mnist-autoencoders", name=cfg.run_name,
           config=OmegaConf.to_container(cfg, resolve=True))
vae = VAE(cfg).to(device)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jess-price/.netrc.
wandb: Currently logged in as: jessprice144 (jessprice144-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# Now i need to define the model and an optimiser
optimiser = torch.optim.Adam(vae.parameters(), lr=cfg.training.lr, momentum=cfg.training.momentum)

In [6]:
# Now i need to define the training loop and, inside, the loss function
for epoch in range(cfg.training.epochs):
    pbar = tqdm(train_loader, desc=f'epoch: {epoch}')
    losses = 0
    for x_train, _ in pbar:
        optimiser.zero_grad()
        x = x_train.to(device)
        x_hat = vae(x)
        loss = vae.calculate_loss(x_hat, x)
        loss.backward()
        losses += loss.item()
        optimiser.step()
    wandb.log({"train/loss": losses/len(train_loader)}, step=epoch)
    with torch.inference_mode():
        testpbar = tqdm(test_loader, leave=False)
        losses = 0
        for x_test, _ in testpbar:
            x = x_test.to(device)
            x_hat = vae(x)
            loss = vae.calculate_loss(x_hat, x)
            losses += loss.item()
        wandb.log({"test/loss": losses/len(test_loader)}, step=epoch)
wandb.finish()

epoch: 0:   0%|                                                                                                                                                        | 0/938 [00:00<?, ?it/s]/home/jess-price/Documents/mnist-autoencoders/src/mnist_autoencoders/models/VAE.py:29: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  latent_log_nrml = F.log_softmax(latent_nrml)
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 157/157 [00:00<00:00, 326.61it/s]


test/loss,█▃▂▂▂▂▁▁▁▁
train/loss,█▄▂▂▂▂▁▁▁▁
test/loss,0.64342
train/loss,0.64478


# Results

In [7]:
# Here I shall test the model visually, passing an image through,
# denormalising the result and rendering it with pillow
data = next(iter(train_loader))[0][1].to(device)

output = vae(torch.reshape(data, (1,1,28,28)))[0][0]
restored_output = output_to_image(output)
img_hat = Image.fromarray(restored_output.cpu().detach().numpy())

restored_data = output_to_image(data)[0]
img = Image.fromarray(restored_data.cpu().numpy())

In [8]:
img

In [9]:
img_hat

In [10]:
# torch.save(vae.state_dict(), '../models/VAE_128/checkpoint_epoch_10.pt')